# Mini Project — Scraping NHL Hockey Statistics

## Project Overview

In this mini project we will scrape **NHL team season statistics** from:

> https://www.scrapethissite.com/pages/forms/

This is a purpose-built scraping practice site. The data is structured in a paginated table and contains real NHL statistics.

We will collect the following fields for every team across all pages:

| Field | Description |
|---|---|
| Team Name | NHL franchise name |
| Year | Season year |
| Wins | Games won |
| Losses | Games lost |
| OT Losses | Overtime losses |
| Win % | Winning percentage |
| Goals For | Total goals scored |
| Goals Against | Total goals conceded |
| Goal Difference | Goals For minus Goals Against |

---

## Project Steps

1. Import libraries
2. Scrape and inspect a single page
3. Extract the table headers
4. Extract table rows from one page
5. Demonstrate pagination
6. Full scraping pipeline across all pages
7. Build the DataFrame
8. Clean the data
9. Analyse the dataset
10. Export to CSV

## Step 1 — Import Libraries

In [1]:
## import necessary library

import requests
from bs4 import BeautifulSoup

## Step 2 — Scrape and Inspect a Single Page

We start by downloading and parsing the first page.


The page contains a table of NHL team statistics. Each row represents one team's performance in a given season.

Pagination is controlled via a `?page_num=` query parameter in the URL:

```
https://www.scrapethissite.com/pages/forms/?page_num=1
https://www.scrapethissite.com/pages/forms/?page_num=2
```

In [3]:
# Base URL — we will append ?page_num= to paginate
base_url = "https://www.scrapethissite.com/pages/forms/"

# Request page 1
page_1_url = base_url + '?page_num=1'
response = requests.get(page_1_url)
soup = BeautifulSoup(response.content, 'html.parser')



In [4]:
response.status_code

200

In [5]:
soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<title>Hockey Teams: Forms, Searching and Pagination | Scrape This Site | A public sandbox for learning web scraping</title>
<link href="/static/images/scraper-icon.png" rel="icon" type="image/png"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<meta content="Browse through a database of NHL team stats since 1990. Practice building a scraper that handles common website interface components." name="description"/>
<link crossorigin="anonymous" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" rel="stylesheet"/>
<link href="https://fonts.googleapis.com/css?family=Lato:400,700" rel="stylesheet" type="text/css"/>
<link href="/static/css/styles.css" rel="stylesheet" type="text/css"/>
<meta content="noindex" name="robo

## Step 3 — Extract Table Headers

The statistics are stored inside an HTML table.

We first extract the **column headers** from the `<thead> or <th>` section so we know what each column represents.

In [6]:
# Locate the stats table
table = soup.find('table')
table

<table class="table">
<tr>
<th>
                            Team Name
                        </th>
<th>
                            Year
                        </th>
<th>
                            Wins
                        </th>
<th>
                            Losses
                        </th>
<th>
                            OT Losses
                        </th>
<th>
                            Win %
                        </th>
<th>
                            Goals For (GF)
                        </th>
<th>
                            Goals Against (GA)
                        </th>
<th>
                            + / -
                        </th>
</tr>
<tr class="team">
<td class="name">
                            Boston Bruins
                        </td>
<td class="year">
                            1990
                        </td>
<td class="wins">
                            44
                        </td>
<td class="losses">
                            2

In [7]:
# Extract column headers from the thead section
header_head = table.find_all("th")
header_head

[<th>
                             Team Name
                         </th>,
 <th>
                             Year
                         </th>,
 <th>
                             Wins
                         </th>,
 <th>
                             Losses
                         </th>,
 <th>
                             OT Losses
                         </th>,
 <th>
                             Win %
                         </th>,
 <th>
                             Goals For (GF)
                         </th>,
 <th>
                             Goals Against (GA)
                         </th>,
 <th>
                             + / -
                         </th>]

In [12]:
column_name = []

for head in header_head:
    column_name.append(head.text.strip())

In [13]:
column_name

['Team Name',
 'Year',
 'Wins',
 'Losses',
 'OT Losses',
 'Win %',
 'Goals For (GF)',
 'Goals Against (GA)',
 '+ / -']

## Step 4 — Extract Table Rows from One Page

Each row represents one team's season record.

We loop through each `<tr>` row and extract the text from each `<td>` cell.

In [14]:
# Extract all rows from the table body
# Inspect the first row
#Inspect the second row

rows = table.find_all('tr')
rows

[<tr>
 <th>
                             Team Name
                         </th>
 <th>
                             Year
                         </th>
 <th>
                             Wins
                         </th>
 <th>
                             Losses
                         </th>
 <th>
                             OT Losses
                         </th>
 <th>
                             Win %
                         </th>
 <th>
                             Goals For (GF)
                         </th>
 <th>
                             Goals Against (GA)
                         </th>
 <th>
                             + / -
                         </th>
 </tr>,
 <tr class="team">
 <td class="name">
                             Boston Bruins
                         </td>
 <td class="year">
                             1990
                         </td>
 <td class="wins">
                             44
                         </td>
 <td class="losses">
          

In [17]:
second_row = rows[1]
cells = [td.text.strip() for td in second_row.find_all('td')]
cells

['Boston Bruins', '1990', '44', '24', '', '0.55', '299', '264', '35']

## Step 5 — Demonstrate Pagination

The table spans multiple pages. Pagination links are shown at the bottom of the page.

We can identify the total number of pages by finding all pagination links and checking the last one.

Each page is accessed by incrementing `?page_num=`.

In [18]:
# Locate the pagination section
pagination = soup.find('ul', class_ = 'pagination')
pagination

<ul class="pagination">
<li>
<a href="/pages/forms/?page_num=1">
<strong>
                                    1
                                    </strong>
</a>
</li>
<li>
<a href="/pages/forms/?page_num=2">
                                    
                                    2
                                    
                                </a>
</li>
<li>
<a href="/pages/forms/?page_num=3">
                                    
                                    3
                                    
                                </a>
</li>
<li>
<a href="/pages/forms/?page_num=4">
                                    
                                    4
                                    
                                </a>
</li>
<li>
<a href="/pages/forms/?page_num=5">
                                    
                                    5
                                    
                                </a>
</li>
<li>
<a href="/pages/forms/?page_num=6">
      

In [26]:
page_links = pagination.find_all('a')
page_links

for link in page_links:
    print(link.text.strip(), link['href'])
    

1 /pages/forms/?page_num=1
2 /pages/forms/?page_num=2
3 /pages/forms/?page_num=3
4 /pages/forms/?page_num=4
5 /pages/forms/?page_num=5
6 /pages/forms/?page_num=6
7 /pages/forms/?page_num=7
8 /pages/forms/?page_num=8
9 /pages/forms/?page_num=9
10 /pages/forms/?page_num=10
11 /pages/forms/?page_num=11
12 /pages/forms/?page_num=12
13 /pages/forms/?page_num=13
14 /pages/forms/?page_num=14
15 /pages/forms/?page_num=15
16 /pages/forms/?page_num=16
17 /pages/forms/?page_num=17
18 /pages/forms/?page_num=18
19 /pages/forms/?page_num=19
20 /pages/forms/?page_num=20
21 /pages/forms/?page_num=21
22 /pages/forms/?page_num=22
23 /pages/forms/?page_num=23
24 /pages/forms/?page_num=24
» /pages/forms/?page_num=2


In [28]:
# Print all available page numbers

# Confirm page 2 loads correctly
page_2 = requests.get(base_url + '?page_num=2')
page_2.status_code

200

In [29]:
page_2_soup = BeautifulSoup(page_2.content, 'html.parser')
page2_rows = page_2_soup.find('table').find_all('tr')


## Step 6 — Full Scraping Pipeline

We now loop through all pages and extract every row from every table.

We keep requesting the next page number until the page returns no table rows, which signals we have gone past the last page.

In [43]:
# Scrape through all pages
import time 

all_rows = []
page_num = 1

while True:
    
    # Request to the Web and Parsing the HTML
    url = base_url + f'?page_num={page_num}'
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    
    #Locate the table
    table = soup.find('table')
    
    
    if not table:
        print(f"No Table found on page {page_num} - Stopping the Scraping")
        break
    
    
    # Locate the Rows on that table
    rows = table.find_all('tr')
    
    # If only header row exists, stop
    if len(rows)<=1:
        print(f'Only Header found on page {page_num} - Stopping')
        break
    
    # headers 
    headers = [th.text.strip() for th in table.find_all('th')]
 
    #Skip first row(header)
    for row in rows[1:]:
        cells = [td.text.strip() for td in row.find_all('td')]
        
        if len(cells) == len(headers):
            all_rows.append(dict(zip(headers, cells)))
            
    print(f'Page {page_num} : {len(rows)- 1} rows scraped')
    page_num += 1
    
    time.sleep(0.5)
    
print(f' Total Records Scraped: {len(all_rows)}')


Page 1 : 25 rows scraped
Page 2 : 25 rows scraped
Page 3 : 25 rows scraped
Page 4 : 25 rows scraped
Page 5 : 25 rows scraped
Page 6 : 25 rows scraped
Page 7 : 25 rows scraped
Page 8 : 25 rows scraped
Page 9 : 25 rows scraped
Page 10 : 25 rows scraped
Page 11 : 25 rows scraped
Page 12 : 25 rows scraped
Page 13 : 25 rows scraped
Page 14 : 25 rows scraped
Page 15 : 25 rows scraped
Page 16 : 25 rows scraped
Page 17 : 25 rows scraped
Page 18 : 25 rows scraped
Page 19 : 25 rows scraped
Page 20 : 25 rows scraped
Page 21 : 25 rows scraped
Page 22 : 25 rows scraped
Page 23 : 25 rows scraped
Page 24 : 7 rows scraped
Only Header found on page 25 - Stopping
 Total Records Scraped: 582


## Step 7 — Build the DataFrame

We convert the list of row dictionaries into a Pandas DataFrame.

In [44]:
# Build the DataFrame 
import pandas as pd

df = pd.DataFrame(all_rows)

In [45]:
df.head()

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25


In [46]:
df.shape

(582, 9)

In [47]:
df.columns

Index(['Team Name', 'Year', 'Wins', 'Losses', 'OT Losses', 'Win %',
       'Goals For (GF)', 'Goals Against (GA)', '+ / -'],
      dtype='str')

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Team Name           582 non-null    str  
 1   Year                582 non-null    str  
 2   Wins                582 non-null    str  
 3   Losses              582 non-null    str  
 4   OT Losses           582 non-null    str  
 5   Win %               582 non-null    str  
 6   Goals For (GF)      582 non-null    str  
 7   Goals Against (GA)  582 non-null    str  
 8   + / -               582 non-null    str  
dtypes: str(9)
memory usage: 41.1 KB


## Step 8 — Clean the Data

All values are currently strings. We convert numeric columns to the correct data types:

- **Year, Wins, Losses, OT Losses, Goals For, Goals Against** → integer
- **Win %** → float
- **Goal Difference** → calculated column (Goals For minus Goals Against)

In [ ]:
# Convert numeric columns
int_cols   = ['Year', 'Wins', 'Losses', 'OT Losses', 'Goals For (GF)', 'Goals Against (GA)']
float_cols = ['Win %']

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Add Goal Difference column
df['Goal Difference'] = df['Goals For (GF)'] - df['Goals Against (GA)']

print('Data types after cleaning:')
print(df.dtypes)
df.head()

Data types after cleaning:
Team Name                 str
Year                    Int64
Wins                    Int64
Losses                  Int64
OT Losses               Int64
Win %                 float64
Goals For (GF)          Int64
Goals Against (GA)      Int64
+ / -                     str
Goal Difference         Int64
dtype: object


,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -,Goal Difference
0,Boston Bruins,1990,44,24,<NA>,0.550,299,264,35,35
1,Buffalo Sabres,1990,31,30,<NA>,0.388,292,278,14,14
2,Calgary Flames,1990,46,26,<NA>,0.575,344,263,81,81
3,Chicago Blackhawks,1990,49,23,<NA>,0.613,284,211,73,73
4,Detroit Red Wings,1990,34,38,<NA>,0.425,273,298,-25,-25


## Step 9 — Analyse the Dataset

With a clean dataset we can answer some interesting questions about NHL history.

In [50]:
# Most wins in a single season
df.sort_values('Wins', ascending=False).head(3)

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -,Goal Difference
126,Detroit Red Wings,1995,62,13,<NA>,0.756,325,181,144,144
382,Detroit Red Wings,2005,58,16,8,0.707,305,209,96,96
58,Pittsburgh Penguins,1992,56,21,<NA>,0.667,367,268,99,99


In [51]:
# Teams with the best average win percentage across all seasons
df.groupby('Team Name')['Win %'].mean().sort_values(ascending=False).head(1)

Team Name
Detroit Red Wings    0.586
Name: Win %, dtype: float64

In [ ]:
# Average goals scored per season over the years (league trend)

## Step 10 — Export to CSV

We save the final cleaned dataset to a CSV file.

In [52]:
df.head()

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -,Goal Difference
0,Boston Bruins,1990,44,24,<NA>,0.550,299,264,35,35
1,Buffalo Sabres,1990,31,30,<NA>,0.388,292,278,14,14
2,Calgary Flames,1990,46,26,<NA>,0.575,344,263,81,81
3,Chicago Blackhawks,1990,49,23,<NA>,0.613,284,211,73,73
4,Detroit Red Wings,1990,34,38,<NA>,0.425,273,298,-25,-25


In [53]:
# Save to CSV
df.to_csv('nhl_stats.csv', index = False)